# S&P Global Ratings — Ações de Rating (produção)

Notebook próprio (não encaixa nos 3 dispatchers genéricos — o dado real
vem de uma API JSON oculta atrás de WAF, não de HTML pra raspar nem de
PDF). Ver `ingestores/REGULATORIO/teste_spglobal.ipynb` para o notebook
de teste da Fase 1 que descobriu o endpoint, o mecanismo de
autenticação anônima (cookie `system_account_token` já entregue pelo
servidor no `GET` da página, sem precisar de navegador) e confirmou que
o WAF só bloqueia fingerprint de automação de navegador — `curl_cffi`
com impersonation passa direto.

Filtro fixo: **Últimos 7 dias** (não 24h, pra não perder fim de semana)
× **Infraestrutura** (`rd5Group=infrastructure`) × **Brasil**
(`countryName=BRA`). Sem filtro de tipo de ação (todas). Dado
estritamente tabular — sem validação de tamanho mínimo de texto (não se
aplica), salvo em **CSV** no Volume (não `.txt`), diferente do resto do
pipeline.

Dedup: a API não devolve URL nem ID estável por item (o campo `id` do
JSON é só a posição dentro da página, reseta a cada página/execução) —
usa um ID sintético (hash de `entityId+ratingActionDate+ratingType+
class+ratingTo+actionName`) como chave do manifesto
(`carregar_manifesto()`/`salvar_manifesto()`, mesmo padrão do resto do
projeto).

In [0]:
%pip install --quiet httpx curl_cffi
dbutils.library.restartPython()

In [0]:
import os
import csv
import json
import time
import random
import hashlib
from datetime import datetime, timezone
from typing import Optional

from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

SOURCE_ID = "sp_ratings_actions"
SOURCE_DESCRICAO = "Linked from S&P Global Ratings — Ações de Rating (Infraestrutura, Brasil)"
PAGINA_URL = "https://www.spglobal.com/ratings/pt/regulatory/ratings-actions"
API_URL = "https://api.use1.prod.ratings.spglobal.com/spcom-spratingsdisclosureapi/extoauthv2/getRatingActionsRequest"
API_KEY = "992ac094-a102-4b9c-ac6e-64be3e68b6cf"

# Filtro fixo desta fonte -- ver introdução. numberOfDays=7 (não 24h) pra
# não perder histórico de fim de semana; rd5Group/countryName confirmados
# na Fase 1 (teste_spglobal.ipynb) rodando contra a API real.
NUMERO_DIAS = "7"
SETOR = "infrastructure"
PAIS = "BRA"
ITENS_POR_PAGINA = 25
MAX_PAGINAS = 10  # trava de segurança -- historico observado nao chega perto disso

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}"
os.makedirs(PASTA_DESTINO, exist_ok=True)

PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)
CAMINHO_MANIFESTO = os.path.join(PASTA_MANIFESTOS, f"{SOURCE_ID}_processados.json")

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

# Colunas do CSV -- mesmos campos nativos da API (ver Fase 1), sem
# renomear pra ficar auditável contra a resposta crua.
COLUNAS_CSV = [
    "id_sintetico", "ratingActionDate", "actionTypeCode", "entityId",
    "sourceProvidedName", "actionLevelIndicator", "actionName", "sectorCode",
    "class", "ratingFrom", "ratingTo", "cwolFrom", "cwolTo", "ratingType",
    "maturityDate",
]

In [0]:
# =============================================================================
# Helpers
# =============================================================================

def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception as e:
        print(f"[manifesto] falha ao carregar ({e}); iniciando vazio.")
        return set()


def salvar_manifesto(caminho: str, ids: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(ids), f, ensure_ascii=False, indent=2)


def id_sintetico(item: dict) -> str:
    """A API não devolve URL nem ID estável por item -- o campo "id" do
    JSON é só a posição dentro da página (1..N), reseta a cada página e
    a cada execução (ver Fase 1, teste_spglobal.ipynb). Constrói uma
    chave estável a partir dos campos de conteúdo pra servir de chave de
    dedup no manifesto."""
    partes = [
        str(item.get("entityId") or ""),
        str(item.get("ratingActionDate") or ""),
        str(item.get("ratingType") or ""),
        str(item.get("class") or ""),
        str(item.get("ratingTo") or ""),
        str(item.get("actionName") or ""),
    ]
    return hashlib.md5("|".join(partes).encode("utf-8")).hexdigest()


def obter_sessao_autenticada(tentativas: int = 4):
    """GET simples na página (curl_cffi, sem navegador) já basta -- o
    servidor entrega o cookie system_account_token no proprio SSR
    (ver Fase 1). Devolve (session, token) ou (None, None)."""
    headers = {"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"}

    for tentativa in range(1, tentativas + 1):
        impersonate = IMPERSONATE_PROFILES[(tentativa - 1) % len(IMPERSONATE_PROFILES)]
        session = cffi_requests.Session()
        try:
            resp = session.get(PAGINA_URL, headers=headers, impersonate=impersonate, timeout=HTTP_TIMEOUT)
            token = session.cookies.get("system_account_token")
            if resp.status_code == 200 and token:
                return session, token
            print(f"  [auth {impersonate} tent {tentativa}/{tentativas}] status={resp.status_code} token={'ok' if token else 'ausente'}")
        except Exception as e:
            print(f"  [auth {impersonate} tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None, None

In [0]:
# =============================================================================
# Etapa 1 — Listar ações de rating (com paginação)
# =============================================================================

def _buscar_pagina(session, token, pagina: int, tentativas: int = 3) -> Optional[dict]:
    headers = {
        "User-Agent": USER_AGENT,
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Origin": "https://www.spglobal.com",
        "Referer": PAGINA_URL,
    }
    payload = {
        "actionType": "",
        "countryName": PAIS,
        "jpSectorWebId": "",
        "locale": "pt_LA",
        "numberOfDays": NUMERO_DIAS,
        "pageLength": str(ITENS_POR_PAGINA),
        "pageNumber": str(pagina),
        "rd5Group": SETOR,
        "urlParam": "",
    }

    for tentativa in range(1, tentativas + 1):
        try:
            resp = session.post(API_URL, params={"apikey": API_KEY}, headers=headers,
                                 json=payload, impersonate="chrome120", timeout=HTTP_TIMEOUT)
            if resp.status_code == 200:
                return resp.json()
            print(f"  [pagina {pagina} tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [pagina {pagina} tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.0))

    return None


def listar_ratings_actions(session, token) -> Optional[list[dict]]:
    """Devolve None se a primeira página falhar (fonte indisponível);
    lista parcial se falhar numa página seguinte (mantém o que já
    coletou, mesmo padrão de ingest-news-ons.ipynb)."""
    itens = []

    for pagina in range(1, MAX_PAGINAS + 1):
        dados = _buscar_pagina(session, token, pagina)
        if dados is None:
            if pagina == 1:
                return None
            print(f"  -> falha ao baixar página {pagina}; seguindo com o que já tem.")
            break

        bloco = dados.get("RatingAction") or []
        itens.extend(bloco)

        total = dados.get("totalNumberOfRecords", 0)
        print(f"  página {pagina}: {len(bloco)} itens (total reportado: {total}).")

        if len(bloco) < ITENS_POR_PAGINA or len(itens) >= total:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
# =============================================================================
# Etapa 2 — Salvar em CSV estruturado (não .txt -- dado tabular, sem texto corrido)
# =============================================================================

def salvar_csv(pasta: str, itens_novos: list[dict]) -> tuple[str, str]:
    sufixo = hashlib.md5(f"{HOJE}-{len(itens_novos)}-{time.time()}".encode()).hexdigest()[:8]
    nome_base = f"{SOURCE_ID}_{HOJE}_{sufixo}"
    caminho_csv = os.path.join(pasta, f"{nome_base}.csv")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=COLUNAS_CSV)
        writer.writeheader()
        for item in itens_novos:
            linha = {coluna: item.get(coluna, "") for coluna in COLUNAS_CSV if coluna != "id_sintetico"}
            linha["id_sintetico"] = item["id_sintetico"]
            writer.writerow(linha)

    metadados = {
        "source_id": SOURCE_ID,
        "title": f"S&P Global Ratings — Ações de Rating ({HOJE})",
        "description": SOURCE_DESCRICAO,
        "url": PAGINA_URL,
        "date": HOJE,
        "published_at": HOJE,
        "formato": "csv",
        "qtd_registros": len(itens_novos),
        "filtros": {"numberOfDays": NUMERO_DIAS, "rd5Group": SETOR, "countryName": PAIS},
    }
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_csv, caminho_json

In [0]:
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução. Posicionado aqui (depois de todo o
# resto do notebook já definido, logo antes da célula de execução) --
# não logo após o restartPython(), mesmo padrão já usado em ARTEMIG, pra
# evitar problema de timing.
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"

In [0]:
# =============================================================================
# Execução
# =============================================================================

try:
    ja_processados = carregar_manifesto(CAMINHO_MANIFESTO)

    session, token = obter_sessao_autenticada()

    if not session or not token:
        print("=== falha ao obter sessão/token autenticado. ===")
        atualizar_status_fonte(
            source_id=SOURCE_ID, sucesso=False, docs_capturados=0,
            erro="falha ao obter system_account_token",
        )
    else:
        itens = listar_ratings_actions(session, token)

        if itens is None:
            print("=== falha ao baixar a primeira página de ações de rating. ===")
            atualizar_status_fonte(
                source_id=SOURCE_ID, sucesso=False, docs_capturados=0,
                erro="download da primeira página falhou",
            )
        else:
            for item in itens:
                item["id_sintetico"] = id_sintetico(item)

            itens_novos = [i for i in itens if i["id_sintetico"] not in ja_processados]
            print(f"{len(itens)} ações de rating na janela, {len(itens_novos)} novas.")

            if itens_novos:
                caminho_csv, caminho_json = salvar_csv(PASTA_DESTINO, itens_novos)
                print(f"  -> salvo em {caminho_csv}")
                print(f"  -> metadados em {caminho_json}")
                ja_processados.update(i["id_sintetico"] for i in itens_novos)
                salvar_manifesto(CAMINHO_MANIFESTO, ja_processados)
            else:
                print("  -> nada novo pra salvar nesta execução.")

            print(f"\n=== {len(itens_novos)} ação(ões) de rating nova(s) salva(s). ===")
            atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=len(itens_novos))

except Exception as e:
    print(f"=== ERRO GERAL: {e} ===")
    atualizar_status_fonte(source_id=SOURCE_ID, sucesso=False, docs_capturados=0, erro=str(e))

print("\n=== Fim. ===")